In [1]:

# Imports and setup

import pandas as pd
import numpy as np
from scipy import stats

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

In [2]:

# Load cleaned data from notebook 01

client_table = pd.read_csv("/Users/akashkumarsamantray/BI_Agent/bi-insight-agent/client_table_clean.csv")
web = pd.read_csv("/Users/akashkumarsamantray/BI_Agent/bi-insight-agent/web_events_clean.csv", parse_dates=['date_time'])

print("client_table:", client_table.shape)
print("web:", web.shape)

client_table: (50487, 10)
web: (317235, 6)


In [3]:

# CELL 3 — Completion rate by Variation (headline metric)

# Reuse the furthest-step-reached logic from notebook 01 to flag completion

step_order = {'start': 0, 'step_1': 1, 'step_2': 2, 'step_3': 3, 'confirm': 4}
web['step_rank'] = web['process_step'].map(step_order)

furthest_step = (
    web.groupby('client_id')['step_rank']
    .max()
    .reset_index()
    .rename(columns={'step_rank': 'furthest_step_rank'})
)

furthest_step = furthest_step.merge(client_table[['client_id', 'Variation']], on='client_id', how='left')
furthest_step['completed'] = furthest_step['furthest_step_rank'] == 4

completion_by_group = furthest_step.groupby('Variation')['completed'].mean()
print("Completion rate by group:")
print(completion_by_group)

# Chi-square test — is the completion rate difference statistically significant,
# or could it just be random noise between groups?
contingency = pd.crosstab(furthest_step['Variation'], furthest_step['completed'])
chi2, p_value, dof, expected = stats.chi2_contingency(contingency)
print(f"\nChi-square p-value: {p_value:.5f}")
print("Significant at 95% confidence" if p_value < 0.05 else "Not significant")

Completion rate by group:
Variation
Control    0.655785
Test       0.692927
Name: completed, dtype: float64

Chi-square p-value: 0.00000
Significant at 95% confidence


In [4]:

# Time spent in the process, by Variation

# For each visit, get the time from first event to last event 

# this is the "surface" metric that looked good for Test in the original analysis

visit_duration = (
    web.groupby(['client_id', 'visit_id'])['date_time']
    .agg(['min', 'max'])
    .reset_index()
)
visit_duration['duration_sec'] = (visit_duration['max'] - visit_duration['min']).dt.total_seconds()

# Average duration per client across their visits
client_duration = visit_duration.groupby('client_id')['duration_sec'].mean().reset_index()
client_duration = client_duration.merge(client_table[['client_id', 'Variation']], on='client_id', how='left')

duration_by_group = client_duration.groupby('Variation')['duration_sec'].mean()
print("Average process duration (seconds) by group:")
print(duration_by_group)

# T-test — is the time difference between groups statistically meaningful?
test_durations = client_duration.loc[client_duration['Variation'] == 'Test', 'duration_sec'].dropna()
control_durations = client_duration.loc[client_duration['Variation'] == 'Control', 'duration_sec'].dropna()

t_stat, p_value = stats.ttest_ind(test_durations, control_durations, equal_var=False)
print(f"\nT-test p-value: {p_value:.5f}")
print("Significant at 95% confidence" if p_value < 0.05 else "Not significant")

Average process duration (seconds) by group:
Variation
Control    293.764519
Test       328.362727
Name: duration_sec, dtype: float64

T-test p-value: 0.00000
Significant at 95% confidence


In [5]:

# Backward navigation / friction signal (the deeper finding)

# This is the key check: did Test group clients move backward through steps more often?

# That's a friction signal hidden underneath the "Test completes more" headline result.
web_sorted = web.sort_values(['client_id', 'visit_id', 'date_time'])

# Flag any row where the step_rank is LOWER than the previous row in the same visit 

# that means the client went backward (e.g. step_3 -> step_2)

web_sorted['prev_step_rank'] = web_sorted.groupby(['client_id', 'visit_id'])['step_rank'].shift(1)
web_sorted['went_backward'] = web_sorted['step_rank'] < web_sorted['prev_step_rank']

backward_by_client = web_sorted.groupby('client_id')['went_backward'].any().reset_index()
backward_by_client = backward_by_client.merge(client_table[['client_id', 'Variation']], on='client_id', how='left')

backward_rate_by_group = backward_by_client.groupby('Variation')['went_backward'].mean()
print("Rate of backward navigation (any step repeated) by group:")
print(backward_rate_by_group)

# Chi-square test on the backward-navigation flag

contingency_backward = pd.crosstab(backward_by_client['Variation'], backward_by_client['went_backward'])
chi2, p_value, dof, expected = stats.chi2_contingency(contingency_backward)
print(f"\nChi-square p-value: {p_value:.5f}")
print("Significant at 95% confidence" if p_value < 0.05 else "Not significant")

Rate of backward navigation (any step repeated) by group:
Variation
Control    0.260860
Test       0.334149
Name: went_backward, dtype: float64

Chi-square p-value: 0.00000
Significant at 95% confidence


In [6]:

# Segment cut: Age (does the effect differ by age group?)

# Split into younger/older clients using the median age as the cutoff

median_age = client_table['clnt_age'].median()
client_table['age_group'] = np.where(client_table['clnt_age'] >= median_age, 'Older', 'Younger')

age_completion = furthest_step.merge(
    client_table[['client_id', 'age_group']], on='client_id', how='left'
)

completion_by_age_variation = age_completion.groupby(['age_group', 'Variation'])['completed'].mean()
print("Completion rate by age group and Variation:")
print(completion_by_age_variation)

Completion rate by age group and Variation:
age_group  Variation
Older      Control      0.638259
           Test         0.670604
Younger    Control      0.674007
           Test         0.715185
Name: completed, dtype: float64


In [7]:

# Segment cut: Tenure (do newer vs longer-tenured clients react differently?)

median_tenure = client_table['clnt_tenure_yr'].median()
client_table['tenure_group'] = np.where(
    client_table['clnt_tenure_yr'] >= median_tenure, 'Long tenure', 'Short tenure'
)

tenure_completion = furthest_step.merge(
    client_table[['client_id', 'tenure_group']], on='client_id', how='left'
)

completion_by_tenure_variation = tenure_completion.groupby(['tenure_group', 'Variation'])['completed'].mean()
print("Completion rate by tenure group and Variation:")
print(completion_by_tenure_variation)

Completion rate by tenure group and Variation:
tenure_group  Variation
Long tenure   Control      0.650099
              Test         0.684638
Short tenure  Control      0.661840
              Test         0.701427
Name: completed, dtype: float64
